In [ ]:
# import fitz

# def read_pdf(pdf_path):

#     doc = fitz.open(pdf_path)

#     full_text = ""

#     for page_num in range(len(doc)):
#         page = doc.load_page(page_num)
#         text = page.get_text()

#         full_text += text+"\n"

#     doc.close()

#     return full_text

# if __name__ == "__main__":

#     pdf_path = "resume.pdf"

#     resume_text = read_pdf(pdf_path)

#     print(resume_text)


In [19]:
import fitz

def read_pdf(pdf_path):

    doc = fitz.open(pdf_path)

    text_blocks = []

    for page in doc:
        blocks = page.get_text("blocks")

        blocks.sort(key=lambda b:(b[1],b[0]))

        for block in blocks:
            text_blocks.append(block[4])

    doc.close

    return "\n".join(text_blocks)

# if __name__ =="__main__":
#     pdf_path = "resume_2.pdf"
#     resume_text = read_pdf(pdf_path)

#     print(resume_text)


In [42]:
import fitz

def read_pdf(pdf_path):

    doc = fitz.open(pdf_path["pdf_path"])

    text_blocks = []

    for page in doc:
        blocks = page.get_text("blocks")

        blocks.sort(key=lambda b:(b[1],b[0]))

        for block in blocks:
            text_blocks.append(block[4])

    doc.close

    return "\n".join(text_blocks)

In [ ]:
agent1_pompt="""
You are an expert resume reader. You are given text extracted from a resume. 
REVIEW, ANALYZE and EXTRACT the following information in JSON SCHEMA provided below. DO NOT EXPLAIN ANYTHING.
Candidates SKILLS, PROJECTS and work Discription is very important. ANALYZE properly and fill this information wisely.
DO NOT PUT SOMETHING THAT IS NOT IN THE RESUME TEXT. If there are multiple companies, return all of them.
{{
"name":"",
"last_name":"",
"address":"",
"highest_qualification":"",
"education":{{
    "bachelors_marks_percent":"",
    "masters_marks_percent":"",
}},
"experience":{{
    "number_of_companies":"",
    "companies":[
        {{
        "company_name":"",
        "role":"",
        "role_description":"",
        "projects_overview":""
        }}
    ]
}},
"skills":{{
    "technical_skills":[],
    "languages_used":[],
    "tools_used":[]
}},
"certifications":[],
"awards":[],
"patents":[],
"publications":[]
}}

Resume Text:
{text_input}

"""

In [ ]:
# agent2_pompt="""
# You are an expert hiring evaluation agent. You are given resume data in JSON format and job discription.
# ANALYZE and EVALUDATE the candidate based on resume data and job description. Evaluate the candidate based on the following criteria and retun in JSON Schema:
# Provide strong and weak points. 5 points summary should provide justification of scores in CRISP and CLEAR points.
# NO NOT EXPLAIN ABOUT THE CRITERIA. JUST RETURN THE JSON SCHEMA.
# {{
# "relevant_experience_score": 0 to 100 %,
# "skill_match_score":0 to 100 %,
# "relevant_projects_score":0 to 100 %,
# "professional_or_education_gap":YES/NO/NOT FOUND,
# "5_points_summary_of_profile":[]
# "overall_score":0 to 100 %
# }}

# JOB DESCRIPTION:
# {job_description}

# RESUME JSON:
# {resume_data}

# """

In [41]:
agent2_prompt = """
You are an expert AI hiring evaluation agent.

You will receive:
1. Resume data in JSON format
2. A job description

Your task is to analyze the resume and evaluate the candidate's suitability for the job.

Scoring Guidelines:
- relevant_experience_score → how closely the candidate's work experience matches the job requirements.
- skill_match_score → how well the candidate's technical skills match the required skills.
- relevant_projects_score → how relevant the candidate's projects are to the job role.
- professional_or_education_gap → return:
  YES → gap detected
  NO → no gap
  NOT_FOUND → insufficient information

Return ONLY valid JSON using the schema below.
Do NOT include explanations outside the JSON.

JSON Schema:
{{
"relevant_experience_score": 0-100%,
"skill_match_score": 0-100%,
"relevant_projects_score": 0-100%,
"professional_or_education_gap": "YES | NO | NOT_FOUND",
"five_point_profile_summary": [],
"overall_score": 0-100%
}}

Rules:
- The five_point_profile_summary must contain exactly 5 short bullet points explaining strengths or weaknesses.
- Points must be concise and directly related to the job description.

JOB DESCRIPTION:
{job_description}

RESUME JSON:
{resume_data}
"""

In [47]:
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

llm = ChatOllama(
    model = "llama3.1",
    temperature=0
)

pdf_reading = RunnableLambda(read_pdf)

prompt_1 = ChatPromptTemplate.from_template(agent1_pompt)

resume_agent = pdf_reading | prompt_1 | llm | StrOutputParser()

result = resume_agent.invoke({"pdf_path": "Not_Relevant.pdf"})

with open('profile_summary.txt', "w") as f:
    f.write(result)

# print("\n------ Structured Resume ------\n")
# print(result)

In [48]:
job_description = """
AI/ML Engineer (Generative AI)

Required Skills:
Python, Machine Learning, Deep Learning, NLP, LangChain,
LLMs, Vector Databases, RAG pipelines, Docker, AWS

Responsibilities:
Develop Generative AI applications using LLMs.
Build RAG based knowledge assistants.
Deploy ML models in production environments.

Experience:
3-6 years in AI/ML engineering.

Education:
Bachelor or Master in Computer Science or AI related field.
"""

In [49]:
prompt_2 = ChatPromptTemplate.from_template(agent2_prompt)

evaluater_agent = prompt_2 | llm | StrOutputParser()

evaluated_result = evaluater_agent.invoke({"job_description":job_description,"resume_data":result})

with open('evaluation.txt',"w") as f:
    f.write(evaluated_result)